# Σιωπηλή τοπολογία: το έλλειμμα του αστέρα, χωρίς περιγραφή του δικτύου

**Το ερώτημα.** Στα τρία κελιά χαμηλής επικοινωνίας του PD (`no_comm`,
`silence`, `no_sense`) ο αστέρας συνεργάζεται λιγότερο από τον δακτύλιο: ο
δακτύλιος είναι ψηλότερα σε 13 από τα 15 ζεύγη μοντέλου–κελιού (sign test
p = 0,007). Δύο εξηγήσεις:

- **Εμπειρία** (η δική μας): ένα φύλλο του αστέρα έχει έναν μόνο συμπαίκτη, άρα
  ένας αποστάτης είναι ό,τι βλέπει.
- **Περιγραφή** (El Mir et al. 2026): η τοπολογία μετρούσε μόνο όταν οι πράκτορες
  μάθαιναν ποιους ακούν· η αποκάλυψη τους κάνει να σκέφτονται τους αόρατους.

**Το πείραμα.** Ίδια δρομολόγηση, αλλά το prompt δεν λέει τίποτα για το δίκτυο:
ούτε σχήμα, ούτε μέγεθος, ούτε πλήθος γειτόνων. Ο πράκτορας βλέπει στο ιστορικό
του πόσους συμπαίκτες έχει («Neighbor #k chose ...»), άρα η εμπειρία μένει και η
περιγραφή φεύγει. Το σχέδιο και ο κανόνας ανάγνωσης είναι στο
`docs/silent_topology_plan.md`, γραμμένα πριν από τα runs.

| βήμα | δίκτυο | κάρτα 0 | κάρτα 1 | runs | ~ώρες |
|---|---|---|---|---|---|
| 1 | αστέρας | gemma-2-9b, μετά Llama-3.1-8B | Qwen3-4B, gemma-2-2b, Qwen2.5-7B | 75 | 6 |
| 2 | δακτύλιος | ίδια | ίδια | 75 | 6 |

Κάθε μοντέλο τρέχει 3 κελιά × 5 runs, PD, 16 γύροι. Τα δύο βήματα είναι
ανεξάρτητα και τρέχουν σε **χωριστά sessions**, με όποια σειρά θες. Ο κανόνας
διαβάζεται όταν έχουν τελειώσει και τα δύο.

## Ο κανόνας (γραμμένος πριν από τα runs)

Στα 15 ζεύγη μοντέλου–κελιού με το σιωπηλό prompt: k = ζεύγη όπου ο δακτύλιος
συνεργάζεται περισσότερο από τον αστέρα, m = το αντίστροφο (οι ισοπαλίες
βγαίνουν από το test).

- **Το έλλειμμα επιβιώνει** όταν k ≥ 12 και το δίπλευρο sign test δίνει
  p < 0,05: το φέρνει η εμπειρία.
- **Το έλλειμμα χάνεται** όταν k ≤ 9: το έφερνε η περιγραφή.
- Αλλιώς **αδιευκρίνιστο**.

Πρόβλεψη: επιβιώνει. Ο ίδιος κώδικας στα ανακοινωμένα runs δίνει k = 13,
m = 2, p = 0,007, δηλαδή το νούμερο του report.

## Setup

1. Settings → Accelerator → **GPU T4 x2** (δύο κάρτες, ένα μοντέλο στην καθεμία)
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN`, **Attach to notebook** (για Llama και τα δύο gemma)

Άλλαξε **μόνο** το `STEP` (1 = αστέρας, 2 = δακτύλιος) και μετά **Save Version
→ Save & Run All**.

- Τα μοντέλα φορτώνουν ένα-ένα: το δεύτερο ξεκινά μόλις το πρώτο είναι στην κάρτα.
- Κάθε μοντέλο τρέχει δύο δουλειές στη σειρά (`no_comm`, μετά `silence` +
  `no_sense`), και τα αρχεία του σβήνονται από τον δίσκο μετά τη δεύτερη.
- Αν φανεί μόνο μία κάρτα, το notebook σταματά πριν τρέξει οτιδήποτε.
- Οι ώρες βγαίνουν από τα μετρημένα λεπτά ανά run του αστέρα (TRACK_RECORD),
  μισά για κελιά χωρίς μηνύματα από το μοντέλο: ±30%.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU off -- Settings -> Accelerator -> GPU T4 x2'
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))
assert N_GPU >= 2, ('Μία μόνο κάρτα: τα βήματα είναι υπολογισμένα για δύο μοντέλα '
                    'παράλληλα -- Settings -> Accelerator -> GPU T4 x2')

In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

src = open('campaign.py', encoding='utf-8').read()
for flag in ('--silent-topology', '--zip-mirror', '--conditions'):
    assert flag in src, f'Το campaign.py δεν έχει {flag} -- κάνε git push πρώτα.'
assert 'SILENT_TOPOLOGY' in open('prompts.py', encoding='utf-8').read(), \
    'Το prompts.py δεν έχει το σιωπηλό κείμενο -- κάνε git push πρώτα.'
assert 'Model loaded' in open('llm_client.py', encoding='utf-8').read(), \
    'το llm_client.py δεν τυπώνει «Model loaded» -- η κλειδαριά φόρτωσης δεν θα ξεκλείδωνε'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

## Ποιο βήμα

In [ ]:
STEP = 1          # <-- ΜΟΝΟ ΑΥΤΟ ΑΛΛΑΖΕΙΣ: 1 = αστέρας, 2 = δακτύλιος

G2B, Q25 = 'google/gemma-2-2b-it', 'Qwen/Qwen2.5-7B-Instruct'
LLA, Q3, G9B = 'meta-llama/Llama-3.1-8B-Instruct', 'Qwen/Qwen3-4B', 'google/gemma-2-9b-it'

TOPOLOGY = {1: 'star', 2: 'cycle'}[STEP]
NET = {'star': 'αστέρας', 'cycle': 'δακτύλιος'}[TOPOLOGY]
# Κάρτα 0: τα δύο αργά μοντέλα (~3 + ~2,3 h). Κάρτα 1: τα τρία γρήγορα
# (~2,5 + ~1,8 + ~1,3 h). Ίδια κατανομή και στα δύο βήματα.
LANES = ([G9B, LLA], [Q3, G2B, Q25])
HOURS = 6

# Δύο δουλειές ανά μοντέλο: το κλειστό σκέλος του baseline, και τα δύο κελιά
# με κανάλι χωρίς λόγια του μοντέλου. Τα max_tokens του session A, όπως στο πλέγμα.
CELLS = {'no_comm': (['baseline'], ['no_comm']),
         'silence+no_sense': (['silence', 'no_sense'], None)}

def args_for(model, scenarios, conditions):
    a = ['--model', model, '--session', 'A', '--topology', TOPOLOGY, '--silent-topology',
         '--games', 'pd', '--n-runs', '5', '--n-rounds', '16', '--scenarios', *scenarios,
         '--out-root', '/kaggle/working/results', '--zip-mirror', '']
    return a + ['--conditions', *conditions] if conditions else a

from campaign import plan
JOBS, EXPECTED, RESULT_DIR = [], {}, {}
for lane in LANES:
    jobs = []
    for model in lane:
        for k, (cell, (scenarios, conditions)) in enumerate(CELLS.items()):
            name = f"{model.split('/')[-1]} · {cell}"
            p = plan(model, 'A', TOPOLOGY, 5, 16, '/kaggle/working/results', only=scenarios,
                     games=['pd'], conditions=conditions, silent_topology=True)
            EXPECTED[name], RESULT_DIR[name] = p['expected_runs'], p['result_dir']
            jobs.append((name, model, args_for(model, scenarios, conditions), k == len(CELLS) - 1))
    JOBS.append(jobs)
NEEDS_HF = any(m.startswith(('meta-llama/', 'google/')) for lane in LANES for m in lane)

print(f'βήμα {STEP}: {NET}, σιωπηλό prompt, PD, ~{HOURS} h')
for g, jobs in enumerate(JOBS):
    print(f'  κάρτα {g}: ' + '  ->  '.join(f'{n} [{EXPECTED[n]}]' for n, *_ in jobs))
print(f'  σύνολο {sum(EXPECTED.values())} runs')
assert sum(EXPECTED.values()) == 75, 'το βήμα δεν έχει 75 runs -- σταμάτα'

## Ο έλεγχος που κρίνει το πείραμα — κοστίζει μηδέν

Χτίζει τα prompts όλων των πρακτόρων με τον κώδικα που θα τρέξει, στα δύο
σκέλη, και επιβεβαιώνει: ότι έχουν το σιωπηλό κείμενο, ότι δεν έχει μείνει
λέξη για σχήμα ή μέγεθος δικτύου, ότι όλοι οι πράκτορες παίρνουν **το ίδιο**
system prompt (κέντρο και φύλλα δεν ξεχωρίζουν πια), και ότι το μόνο που
αλλάζει σε σχέση με το ανακοινωμένο prompt είναι οι δύο παράγραφοι του δικτύου.

In [ ]:
import re, sys
sys.path.insert(0, '.')
from config import ExperimentConfig, ModelConfig
from engine import make_engine
from prompts import SILENT_TOPOLOGY, SILENT_COMMUNICATION

def system_prompts(condition, silent):
    cfg = ExperimentConfig(game='pd', condition=condition, topology=TOPOLOGY, n_agents=4,
                           model=ModelConfig(provider='local', model_id=Q25),
                           topology_aware_comm_prompt=True, silent_topology=silent)
    return [a._system_prompt() for a in make_engine(cfg).build_agents(client=None)]

LEAK = re.compile(r'star|ring|cycle|central|peripheral|You have \d|one of \d|\b[1-9] neighbor', re.I)
for condition in ('no_comm', 'cheap_talk'):
    silent, told = system_prompts(condition, True), system_prompts(condition, False)
    assert len(set(silent)) == 1, f'{condition}: οι πράκτορες παίρνουν διαφορετικό prompt -- σταμάτα'
    p = silent[0]
    assert SILENT_TOPOLOGY in p, f'{condition}: λείπει το σιωπηλό κείμενο -- σταμάτα'
    if condition == 'cheap_talk':
        assert SILENT_COMMUNICATION in p, 'λείπει η σιωπηλή δρομολόγηση -- σταμάτα'
    leaks = LEAK.findall(p)
    assert not leaks, f'{condition}: έμειναν λέξεις δικτύου {leaks} -- σταμάτα'
    gone = [l for l in told[0].splitlines() if l not in p.splitlines()]
    print(f'{condition}: ίδιο prompt για τους 4, χωρίς λέξεις δικτύου· φεύγουν {len(gone)} γραμμές:')
    for l in gone:
        print('   -', l.strip()[:150])
    assert 1 <= len(gone) <= 2, f'{condition}: άλλαξαν κι άλλα εκτός από το δίκτυο -- σταμάτα'
print(f'\nσιωπηλό prompt σωστό, {NET}, και στα δύο σκέλη')

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token φορτώθηκε')
except Exception as e:
    print('Χωρίς HF_TOKEN:', e)

## Preview — κοστίζει μηδέν

Σε κάθε δουλειά πρέπει να δεις τη σωστή τοπολογία, `games : ['pd']`, τη γραμμή
`NETWORK : not described`, το σωστό `expecting`, και φάκελο που τελειώνει σε
`_silent`. Η εντολή `RUN:` δεν πρέπει να έχει `--zip-mirror`.

In [ ]:
import os, subprocess, sys

ok = True
if NEEDS_HF and not os.environ.get('HUGGINGFACE_API_KEY'):
    ok = False
    print('ΛΕΙΠΕΙ ΤΟ HF_TOKEN -- Llama και gemma είναι gated, θα αποτύχουν σε ~1 λεπτό.')
for g, jobs in enumerate(JOBS):
    for name, model, extra, _ in jobs:
        print('=' * 70)
        print(f'κάρτα {g} · {name}')
        r = subprocess.run([sys.executable, 'campaign.py', *extra, '--dry-run'],
                           capture_output=True, text=True)
        out = r.stdout.strip()
        print(out[:900])
        if r.returncode != 0:
            ok = False
            print('ΣΦΑΛΜΑ:', (r.stderr or r.stdout).strip()[-300:])
        for needle in (f'topology     : {TOPOLOGY}', "games        : ['pd']",
                       'NETWORK      : not described',
                       f'expecting    : {EXPECTED[name]} run files', RESULT_DIR[name]):
            if needle not in out:
                ok = False
                print(f'ΛΕΙΠΕΙ ΑΠΟ ΤΟ ΠΛΑΝΟ: {needle!r}')
        if not RESULT_DIR[name].endswith('_silent'):
            ok = False
            print('Ο ΦΑΚΕΛΟΣ ΔΕΝ ΤΕΛΕΙΩΝΕΙ ΣΕ _silent')
        if '--zip-mirror' in out.split('RUN:')[-1]:
            ok = False
            print('ΤΟ MIRROR ΕΙΝΑΙ ΕΝΕΡΓΟ -- τα δύο μοντέλα θα έγραφαν τα ίδια zip')

assert ok, 'Κάποιο plan απέτυχε -- μη συνεχίσεις.'
print('\n' + '=' * 70)
print('όλα τα πλάνα εντάξει')

## Εκτέλεση: δύο κάρτες, μία ουρά η καθεμία

Κάθε δουλειά είναι ένα `campaign.py` με δικό του log στο `/kaggle/working/logs/`.
Εδώ τυπώνεται η γραμμή `device map` κάθε μοντέλου μόλις φορτώσει, η πρόοδος κάθε
10 λεπτά, κάθε δουλειά που τελειώνει, και ο απολογισμός. Στις 11,5 ώρες ό,τι
τρέχει σταματά, και τα σενάρια που τελείωσαν πακετάρονται παρακάτω.

In [ ]:
import glob, os, shutil, subprocess, sys, threading, time

LOG_DIR, RESULTS = '/kaggle/working/logs', '/kaggle/working/results'
POLL_S, REPORT_S, DEADLINE_H = 30, 600, 11.5
os.makedirs(LOG_DIR, exist_ok=True)

try:
    from huggingface_hub import constants as _hfc
    HF_CACHE = _hfc.HF_HUB_CACHE
except Exception:
    HF_CACHE = os.path.expanduser('~/.cache/huggingface/hub')

def free_gb(path):
    while not os.path.exists(path):
        path = os.path.dirname(path)
    return shutil.disk_usage(path).free / 1e9

# Κάθε μοντέλο μένει στον δίσκο ανάμεσα στις δύο δουλειές του. Ως ~35 GB μαζί
# (gemma-2-9b 18,5 + Qwen3-4B 8, ή Llama 16 + Qwen2.5 15). Με λιγότερο από 38 GB
# ελεύθερα, τα αρχεία σβήνονται μόλις φορτώσει κάθε δουλειά, και η δεύτερη
# δουλειά του μοντέλου τα ξανακατεβάζει (λίγα λεπτά).
FREE = free_gb(HF_CACHE)
KEEP_BETWEEN = FREE >= 38
print(f'ελεύθερος δίσκος {FREE:.0f} GB: '
      + ('κάθε μοντέλο κατεβαίνει μία φορά' if KEEP_BETWEEN else
         'τα αρχεία σβήνονται μετά από κάθε φόρτωση (ξανακατέβασμα στη 2η δουλειά)'))
assert FREE > 22, 'λιγότερα από 22 GB ελεύθερα: ούτε το gemma-2-9b δεν χωρά -- σταμάτα'

def cache_dir(model):
    return os.path.join(HF_CACHE, 'models--' + model.replace('/', '--'))

def runs_done(name):
    d = RESULT_DIR[name]
    return sum(1 for p in glob.glob(f'{d}/**/*.json', recursive=True)
               if not os.path.basename(p).startswith('_') and f'{os.sep}zips{os.sep}' not in p)

def log_has(path, needle):
    try:
        with open(path, encoding='utf-8', errors='replace') as fh:
            return needle in fh.read()
    except FileNotFoundError:
        return False

load_lock = threading.Lock()   # ένα μοντέλο φορτώνει τη φορά
guard = threading.Lock()       # προστατεύει τα state / finished / procs
state, finished, procs = {}, [], {}
t0 = time.time()

def run_lane(gpu, jobs):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1',
               TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='2')
    for i, (name, model, extra, last_of_model) in enumerate(jobs):
        log = os.path.join(LOG_DIR, f"step{STEP}_gpu{gpu}_{i}_{model.split('/')[-1]}.log")
        t = time.time()
        if t - t0 > DEADLINE_H * 3600:
            with guard:
                finished.append((gpu, name, 'SKIPPED (χρόνος)', 0.0, log))
            continue
        try:
            with load_lock:
                with guard:
                    state[gpu] = dict(name=name, model=model, log=log, t0=t, phase='φόρτωση')
                with open(log, 'w', encoding='utf-8') as fh:
                    proc = subprocess.Popen([sys.executable, 'campaign.py', *extra],
                                            stdout=fh, stderr=subprocess.STDOUT, env=env)
                with guard:
                    procs[gpu] = proc
                while proc.poll() is None and not log_has(log, '[local] Model loaded'):
                    time.sleep(10)
                if not KEEP_BETWEEN and proc.poll() is None:
                    shutil.rmtree(cache_dir(model), ignore_errors=True)
            with guard:
                state[gpu]['phase'] = 'runs'
            rc = proc.wait()
            status = 'OK' if rc == 0 else f'FAILED (exit {rc})'
        except Exception as e:   # ποτέ σιωπηλά: η δουλειά μπαίνει στον απολογισμό
            status = f'ERROR {e!r}'[:60]
        if last_of_model:
            shutil.rmtree(cache_dir(model), ignore_errors=True)
        with guard:
            finished.append((gpu, name, status, (time.time() - t) / 60, log))
    with guard:
        state[gpu] = None

threads = []
for gpu, jobs in enumerate(JOBS):
    th = threading.Thread(target=run_lane, args=(gpu, jobs), daemon=True)
    th.start()
    threads.append(th)
    time.sleep(2)   # η κάρτα 0 παίρνει πρώτη την κλειδαριά φόρτωσης

shown_maps, n_reported, last = set(), 0, time.time()
while any(th.is_alive() for th in threads):
    time.sleep(POLL_S)
    if time.time() - t0 > DEADLINE_H * 3600:
        with guard:
            for p in procs.values():
                if p.poll() is None:
                    p.terminate()
    with guard:
        snap = {g: dict(s) for g, s in state.items() if s}
        done = list(finished)
    for g, s in snap.items():
        if s['log'] in shown_maps:
            continue
        try:
            with open(s['log'], encoding='utf-8', errors='replace') as fh:
                line = next((l for l in fh if '[local] device map' in l), None)
        except FileNotFoundError:
            line = None
        if line:
            shown_maps.add(s['log'])
            print(f"κάρτα {g} · {s['name']}: {line.strip()}", flush=True)
    for g, name, st, mins, _ in done[n_reported:]:
        print(f'--> κάρτα {g} · {name}: {st}, {mins:.0f} λεπτά', flush=True)
    n_reported = len(done)
    if time.time() - last >= REPORT_S:
        last = time.time()
        print(f'--- {(last - t0) / 3600:.1f} h ---', flush=True)
        for g, s in sorted(snap.items()):
            print(f"  κάρτα {g}: {s['name']:34s} {s['phase']:8s} "
                  f"{runs_done(s['name']):3d}/{EXPECTED[s['name']]} runs, "
                  f"{(time.time() - s['t0']) / 3600:.1f} h", flush=True)

for th in threads:
    th.join()
for g, name, st, mins, _ in finished[n_reported:]:
    print(f'--> κάρτα {g} · {name}: {st}, {mins:.0f} λεπτά')

print('\n' + '=' * 70)
print('ΑΠΟΛΟΓΙΣΜΟΣ')
for g, name, st, mins, _ in finished:
    print(f'  κάρτα {g}  {st:22s} {mins:6.0f} λ   {name}')
print(f'\nσύνολο {(time.time() - t0) / 3600:.1f} h  (εκτίμηση ~{HOURS} h)')
for g, name, st, mins, log in finished:
    if st != 'OK' and os.path.exists(log):
        print('\n' + '-' * 70)
        print(f'ΑΠΟΤΥΧΙΑ: κάρτα {g} · {name} -- οι τελευταίες γραμμές του {log}:')
        with open(log, encoding='utf-8', errors='replace') as fh:
            print(''.join(fh.readlines()[-25:]))

## Έλεγχος των δεδομένων και πρώτη ματιά

Επιβεβαιώνει ότι κάθε run είναι PD, η σωστή τοπολογία, 16 γύροι, με
`silent_topology` γραμμένο στο config. Τυπώνει ανά κελί τη συνεργασία και τις
άκυρες αποφάσεις, δίπλα στα ανακοινωμένα runs (αστέρας του πλέγματος, δακτύλιος
με διορθωμένο prompt, άλλη περίοδος: μόνο για προσανατολισμό). Μετά τρέχει το
`silent_topology_report.py`, που διαβάζει τον κανόνα μόνο όταν υπάρχουν και τα
δύο δίκτυα.

In [ ]:
import collections, glob, json, os, statistics, subprocess, sys
sys.path.insert(0, '.')
from analysis import summarise_run, scenario_of

# Ανακοινωμένα runs, μέσος όρος run: (αστέρας του πλέγματος, δακτύλιος με
# διορθωμένο prompt). Από το silent_topology_report.py με silent=False.
REF = {
    ('Llama-3.1-8B-Instruct', 'no_comm'): (0.47, 0.51),
    ('Llama-3.1-8B-Instruct', 'no_sense'): (0.48, 0.52),
    ('Llama-3.1-8B-Instruct', 'silence'): (0.33, 0.59),
    ('Qwen2.5-7B-Instruct', 'no_comm'): (0.08, 0.24),
    ('Qwen2.5-7B-Instruct', 'no_sense'): (0.12, 0.14),
    ('Qwen2.5-7B-Instruct', 'silence'): (0.43, 0.72),
    ('Qwen3-4B', 'no_comm'): (0.04, 0.05),
    ('Qwen3-4B', 'no_sense'): (0.92, 0.96),
    ('Qwen3-4B', 'silence'): (0.70, 1.00),
    ('gemma-2-2b-it', 'no_comm'): (0.67, 0.46),
    ('gemma-2-2b-it', 'no_sense'): (0.26, 0.07),
    ('gemma-2-2b-it', 'silence'): (0.04, 0.07),
    ('gemma-2-9b-it', 'no_comm'): (0.08, 0.28),
    ('gemma-2-9b-it', 'no_sense'): (0.09, 0.18),
    ('gemma-2-9b-it', 'silence'): (0.30, 0.52),
}

coop, inval = collections.defaultdict(list), collections.defaultdict(list)
roots = sorted(glob.glob(f'{RESULTS}/*_silent'))
for d in roots:
    for p in glob.glob(f'{d}/**/*.json', recursive=True):
        if os.path.basename(p).startswith('_') or f'{os.sep}zips{os.sep}' in p:
            continue
        rec = json.load(open(p, encoding='utf-8'))
        cfg = rec['config']
        assert cfg['game'] == 'pd', f'όχι PD: {p}'
        assert rec['topology']['type'] == TOPOLOGY, f'λάθος τοπολογία: {p}'
        assert cfg.get('silent_topology') is True, f'ΛΑΘΟΣ PROMPT (όχι σιωπηλό): {p}'
        assert len(rec['history']) == 16, f'{len(rec["history"])} γύροι: {p}'
        model = cfg['model']['model_id'].split('/')[-1]
        cell = 'no_comm' if cfg['condition'] == 'no_comm' else scenario_of(rec)
        s = summarise_run(rec)
        coop[(model, cell)].append(s['coop_rate_overall'])
        inval[(model, cell)].append(s['invalid_rate'])

col = 0 if TOPOLOGY == 'star' else 1
print(f"{'μοντέλο':22s} {'κελί':10s} {'n':>2s} {'σιωπηλό':>8s} {'άκυρα':>6s} {'ανακοινωμένο':>13s}")
for k in sorted(coop):
    c, iv = statistics.fmean(coop[k]), statistics.fmean(inval[k])
    ref = REF.get(k, (float('nan'), float('nan')))[col]
    flag = '  <- >15% άκυρα' if iv > 0.15 else ''
    print(f'{k[0]:22s} {k[1]:10s} {len(coop[k]):2d} {c:8.2f} {iv:6.0%} {ref:13.2f}{flag}')
print(f'\n{sum(len(v) for v in coop.values())} runs από {len(roots)} φακέλους')

r = subprocess.run([sys.executable, 'silent_topology_report.py', '--roots', *roots],
                   capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-1500:])

## Μάζεμα

In [ ]:
import glob, os, shutil

box = f'/kaggle/working/silent_{TOPOLOGY}'
os.makedirs(box, exist_ok=True)
zips = sorted(glob.glob('/kaggle/working/*_sessionA_pd*_silent.zip'))
print(f'{len(zips)} zip, ένα ανά δουλειά που πέρασε τον έλεγχο του campaign.py:')
for z in zips:
    print(f'   {os.path.getsize(z)/1e6:6.1f} MB  {os.path.basename(z)}')
    shutil.copy(z, box)

# Δουλειά που απέτυχε ή κόπηκε δεν έχει zip campaign· τα σενάρια που πρόλαβε να
# τελειώσει έχουν το δικό τους zip μέσα στον φάκελο του μοντέλου.
for g, name, st, mins, log in finished:
    if st != 'OK':
        part = glob.glob(f'{RESULT_DIR[name]}/zips/*.zip')
        if part:
            dst = os.path.join(box, 'partial', os.path.basename(RESULT_DIR[name]))
            os.makedirs(dst, exist_ok=True)
            for z in part:
                shutil.copy(z, dst)
            print(f'   μερικά σενάρια του {name}: {len(part)} zip -> partial/')
shutil.copytree(LOG_DIR, os.path.join(box, 'logs'), dirs_exist_ok=True)

out = shutil.make_archive(f'/kaggle/working/silent_topology_step{STEP}_{TOPOLOGY}', 'zip', box)
print(f'\nκατέβασε αυτό: {out}  ({os.path.getsize(out)/1e6:.1f} MB)')

## Μετά

1. Κατέβασε το `silent_topology_step<N>_<δίκτυο>.zip`. Μέσα έχει ένα zip ανά
   δουλειά: αποσυμπίεσε το καθένα σε φάκελο με το όνομά του (χωρίς `.zip`) μέσα
   στο `diplomatikh/silent_runs/`. Τα runs **δεν** μπαίνουν στο tracker: είναι
   παραλλαγή του prompt, όπως τα `harm_source/`, και το ingest σωστά τα αρνείται.
2. Γράψε στο `TRACK_RECORD.md` τον χρόνο κάθε δουλειάς από τον απολογισμό.
3. Όταν υπάρχουν και τα δύο βήματα, ο κανόνας (από τον φάκελο `diplomatikh`):

```
python cheaptalk_bench/silent_topology_report.py --roots silent_runs/* --announced-roots star_runs/*_star cycle_runs/*_cycle_commfix
```